# FinGPT-QLoRA Interactive Demo

Try the fine-tuned financial sentiment model interactively.

**Model:** Qwen2.5-7B-Instruct + LoRA (v3 completion_only_loss)

**Accuracy:** 83.3% | **ROUGE-L:** 0.981

In [ ]:
!pip install unsloth
!pip install --force-reinstall --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
import os
import subprocess

# Download LoRA adapter via Kaggle API
LORA_PATH = 'outputs/lora_adapter'
os.makedirs(LORA_PATH, exist_ok=True)

print('Downloading LoRA adapter...')
result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'ericwang7717/fingpt-lora-adapter-v3',
     '-p', LORA_PATH, '--unzip'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('Download successful!')
    print(f'Contents of {LORA_PATH}:')
    for f in os.listdir(LORA_PATH):
        print(f'  {f}')
else:
    print(f'Download failed: {result.stderr}')
    raise RuntimeError('Failed to download LoRA adapter')

In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

print(f'Loading model from {LORA_PATH}...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
print(f'Model loaded! VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB')

In [ ]:
SYSTEM_PROMPT = 'You are a financial sentiment analyst. Classify the sentiment of financial text as Positive, Negative, or Neutral. Explain your reasoning concisely.'

def analyze(text):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': f'Analyze the sentiment of this financial text:\n\n{text}'},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(input_ids=input_ids, max_new_tokens=512, do_sample=False)
    return tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)

print('analyze(text) function ready!')

In [ ]:
# === Try it! ===
# Change the text below and run this cell

text = 'Tesla shares surged 12% after record Q3 deliveries'

print(f'Input: {text}')
print('='*60)
print(analyze(text))

In [ ]:
# === Try more examples ===

examples = [
    'Apple reported record revenue of $120 billion, beating analyst expectations by 15%',
    'Fed raises interest rates by 75 basis points, markets tumble on recession fears',
    'Amazon announces 10,000 layoffs in corporate division',
    'Nvidia stock hits all-time high as AI chip demand soars',
    'China\'s GDP growth slows to 3%, lowest in decades',
]

for i, text in enumerate(examples, 1):
    print(f'\nExample {i}: {text[:60]}...')
    print('-'*60)
    print(analyze(text))